# 0-shot to get outcome probs

## Setr Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 500)
print(f"BASE PATH: {BASE_PATH}")

Load model

In [ ]:
## MODEL
model_dir = Path("/lscratch3/models_llama/")
models = []
for idx, entry in enumerate(model_dir.iterdir()):
    print(f"{idx} \t {entry.name}")
    models.append(entry.name)

In [ ]:
CUR_MODEL = models[4]
MODEL_ID = model_dir / CUR_MODEL
print(f"Using model: {CUR_MODEL}")
assert Path(MODEL_ID).exists()

## Run Impact Types (self, parent) in parallel

In [ ]:
# Try 1: just outcomes w/ AUROC > 0.8 on impactme
OUTCOMES = [
    "a1",
    "a2",
    "a3",
    "a4",
    "a5",
    "a6",
    "a7",
    "a8",  # 0.7445
    "b1",  # 0.7821
    "b2",
    "b3",  # 0.7786
    "c1",
    "c2",
    "c3",
    "c4",
    "d1",
    "d2",
    "d3",
    "d4",
    "d5",
    "d6",
    "e1",
    "e2",
    "e3",
    "e4",  # 0.7724- also previously left out bc of splits
    "e5",  # 0.7006- also previously left out bc of splits
    "f1",
    "f2",
    "f3",
    "g1",
    "g2",  # 0.7227
]
CONFIG = {
    "model_id": str(MODEL_ID),
    "temp": 0.6,
}
IMPACT_TYPES = ["self", "parent"]

Run

In [ ]:
import subprocess
import json
import signal


def make_cmd(
    cur_model,
    outcome_list,
    impact_type_list,
    base_df_dir,
    json_config,
    gpu_group,
    outcome_desc_type,
    n_shot,
    output_type,
):
    temp = json_config["temp"]
    log_dir = (
        BASE_PATH
        / "logs"
        / f"shot_{n_shot}"
        / f"temp_{temp}"
        / output_type
        / cur_model
        / outcome_desc_type
    )
    save_dir = (
        BASE_PATH
        / "data"
        / f"w_{n_shot}_shot_probs"
        / f"temp_{temp}"
        / output_type
        / cur_model
        / outcome_desc_type
    )
    cmd = [
        "python",
        "src/shot_controller.py",
        "--gpu-groups",
        *gpu_group,
        "--outcomes",
        *outcome_list,
        "--impact_types",
        *impact_type_list,
        "--llm_config",
        json.dumps(json_config),
        "--base_df_dir",
        str(base_df_dir),
        "--save_dir",
        str(save_dir),
        "--log_dir",
        str(log_dir),
        "--outcome_desc_type",
        outcome_desc_type,
        "--n_shot",
        str(n_shot),
        "--output_type",
        output_type,
    ]
    return cmd

In [ ]:
### Set env
env = os.environ.copy()
env["PYTHONPATH"] = str(BASE_PATH)
if "70B" in CUR_MODEL.upper():
    gpu_groups = [
        "0,1,2,3",
        "4,5,6,7",
    ]
else:
    gpu_groups = [
        "0,1",
        "2,3",
        "4,5",
        "6,7",
    ]

base_df_dir = BASE_PATH / "data" / "cleaned" / "question_answer"
cmd = make_cmd(
    cur_model=CUR_MODEL,
    outcome_list=OUTCOMES,
    impact_type_list=IMPACT_TYPES,
    base_df_dir=base_df_dir,
    json_config=CONFIG,
    gpu_group=gpu_groups,
    outcome_desc_type="original",  # one of original, augment, short
    n_shot=0,  # 3 max
    output_type="proba-limited",  # one of binary, proba, proba-limited
)
proc = subprocess.Popen(
    cmd,
    cwd=BASE_PATH,
    start_new_session=True,
    env=env,
)
print("Controller PID:", proc.pid)

In [ ]:
proc.poll()  # monitor

In [ ]:
os.killpg(proc.pid, signal.SIGTERM)  # gracefull shtudown

In [ ]:
os.killpg(proc.pid, signal.SIGKILL)  # force kill

# Check output

In [ ]:
import_path = (
    BASE_PATH
    / "data"
    / f"w_{0}_shot_probs"
    / f"temp_{0.6}"
    / "proba-limited"
    / CUR_MODEL
    / "original"
    / "parent.parquet"
)
dummy = pd.read_parquet(import_path)
for row in dummy.itertuples():
    print(row.impact_score_parent)
    print(row.open_text_block)
    print(row.open_text_block_0_shot)
    break